# 01. 후보 비교: 진열대 사진 → 제품별 개수

**목적**: 문제정의서의 성공 기준(S1·S2·S4)을 어떤 구성이 넘는지, 성격이 다른 후보를 **같은 사진·같은 정답·같은 지표**로 비교한다. (`모델선정계획.md` 참고)

| 후보 | 용기 찾기 | 제품 구분 | 성격 |
|---|---|---|---|
| A | YOLO11 (COCO 사전학습) | SigLIP 2 | 전용 탐지 |
| A2 | YOLO-World (문장으로 대상 지정) | SigLIP 2 | A의 변형 |
| B | SAM 2 자동 분할 | SigLIP 2 | 범용 분할 |

> **상태: 이 노트북은 작성만 했고 Colab에서 아직 실행해 보지 않았다.** 실행하다 오류가 나면 그 셀의 오류 메시지를 그대로 알려 주면 고친다.

## 규칙 (결과를 보기 전에 정함)
1. 판정 값(임계값, 크기 필터 등)은 **개발셋(쉬움 8장)에서만** 정한다. 평가셋(보통 8 + 어려움 4 = 12장)은 값을 정한 뒤 **한 번만** 돌려 성공 여부를 판정한다.
2. 정한 값은 `results/chosen_params.json`에 먼저 저장하고, 평가셋 결과를 본 뒤에는 바꾸지 않는다.
3. 점수는 반올림 없이 저장하고, 박스를 그린 이미지를 함께 저장한다.
4. 사진을 외부 서비스로 보내지 않는다. API 키를 쓰지 않는다.

**Colab 사용법**: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 → 위에서부터 셀을 차례로 실행.

## 0. 준비

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/skygaul/restock-vision-poc.git"
if not os.path.exists("data/photos"):
    if not os.path.exists("restock-vision-poc"):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("restock-vision-poc")
print(os.getcwd())
print(sorted(os.listdir(".")))

In [ ]:
!pip -q install ultralytics "transformers>=4.49" accelerate

In [ ]:
import sys, json, time, math, platform, datetime, itertools, shutil
from collections import defaultdict
import numpy as np
import torch, torchvision
from PIL import Image, ImageDraw
import transformers, ultralytics

sys.path.insert(0, "scripts")
import evaluate as ev   # 지표 계산 코드(scripts/evaluate.py)를 그대로 쓴다

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    print("[경고] GPU가 아닙니다. 런타임 > 런타임 유형 변경에서 T4 GPU를 선택하세요.")

ENV = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "ultralytics": ultralytics.__version__,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
}
RUN_ID = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
print(json.dumps(ENV, ensure_ascii=False, indent=1), RUN_ID)

## 1. 데이터와 정답

In [ ]:
SEL = ev.load_selection()                       # {"dev": 쉬움 8장, "eval": 보통+어려움 12장, "all": 20장}
GT = ev.read_long_or_wide("data/ground_truth.csv")
PRODUCTS = ["P1", "P2", "P3", "P4", "P5", "P6", "P7", "P8"]

def load_img(fn):
    return Image.open(f"data/photos/{fn}.jpg").convert("RGB")

print({k: len(v) for k, v in SEL.items()})
print("개발셋:", SEL["dev"])
print("평가셋:", SEL["eval"])

## 2. 제품 구분기 (SigLIP 2)
잘라낸 용기 그림 한 장마다 **제품 문장과 얼마나 맞는지**를 점수(sigmoid, 0~1)로 낸다. 점수는 문장마다 따로 계산되므로 임계값을 걸 수 있다(softmax는 쓰지 않는다).

- 8종(P1~P8) 문장 + **기타**(8종과 비슷하지만 다른 제품) 문장 + **배경**(가격표, 선반 등 용기가 아닌 것) 문장을 함께 넣는다.
- 가장 점수가 높은 쪽이 배경이면 버리고, 기타이거나 8종 점수가 `tau`보다 낮으면 기타로 센다.
- 용량(355/500ml)은 문장으로 구분하기 어렵다. 이 한계는 실패 사례로 분석한다.

In [ ]:
from transformers import AutoModel, AutoProcessor

SIGLIP_ID = "google/siglip2-base-patch16-224"
processor = AutoProcessor.from_pretrained(SIGLIP_ID)
siglip = AutoModel.from_pretrained(SIGLIP_ID).to(DEVICE).eval()
if DEVICE == "cuda":
    siglip = siglip.half()

CLASSES = PRODUCTS + ["other", "bg"]
PROMPTS = {
    "P1": ["a green soju glass bottle, Jinro Chamisul", "참이슬 소주 초록색 유리병"],
    "P2": ["a green soju glass bottle, Chum Churum", "처음처럼 소주 초록색 유리병"],
    "P3": ["a blue Cass beer can", "카스 맥주 파란색 캔"],
    "P4": ["a green Terra beer can", "테라 맥주 초록색 캔"],
    "P5": ["a red Coca-Cola Original can", "코카콜라 오리지널 빨간 캔"],
    "P6": ["a red Coca-Cola Zero can with black lettering", "코카콜라 제로 빨간 캔 검정 글씨"],
    "P7": ["a green and white Chilsung Cider can", "칠성사이다 초록색 흰색 캔"],
    "P8": ["a Coca-Cola plastic bottle with a red cap", "코카콜라 페트병 빨간 뚜껑"],
    "other": ["a Pepsi can", "a light blue Cass Light beer can", "a yellow Kelly beer can",
              "a black Chilsung Cider Zero can", "a green Hite or Hanmac beer can",
              "a plastic soju bottle", "a white milk or rice wine bottle",
              "an orange or yellow soft drink bottle", "a Sprite plastic bottle",
              "another beverage can or bottle"],
    "bg": ["a yellow price tag", "an empty shelf edge", "a poster or sign", "the background of a store shelf"],
}
FLAT = [(c, t) for c in CLASSES for t in PROMPTS[c]]
FLAT_CLASS = [c for c, _ in FLAT]
TEXTS = [t.lower() for _, t in FLAT]          # SigLIP 2는 소문자 문장으로 학습됨

@torch.no_grad()
def classify(crops, batch=32):
    """crops(PIL 리스트) -> (N, 클래스 수) 점수. 클래스마다 문장 점수의 최댓값을 쓴다."""
    outs = []
    for i in range(0, len(crops), batch):
        inp = processor(text=TEXTS, images=crops[i:i + batch], padding="max_length",
                        max_length=64, return_tensors="pt")
        inp = {k: (v.to(DEVICE, torch.float16) if (k == "pixel_values" and DEVICE == "cuda") else v.to(DEVICE))
               for k, v in inp.items()}
        logits = siglip(**inp).logits_per_image          # (그림 수, 문장 수)
        outs.append(torch.sigmoid(logits.float()).cpu().numpy())
    if not outs:
        return np.zeros((0, len(CLASSES)))
    s = np.concatenate(outs)
    res = np.zeros((s.shape[0], len(CLASSES)))
    for ci, c in enumerate(CLASSES):
        res[:, ci] = s[:, [j for j, cc in enumerate(FLAT_CLASS) if cc == c]].max(1)
    return res

print("문장 수:", len(TEXTS))

## 3. 용기 찾기 (후보 A, A2, B)
각 방식이 사진마다 **상자(박스)**를 낸다. 상자의 좌표는 원본 사진 기준 픽셀이다.

In [ ]:
from ultralytics import YOLO, YOLOWorld, SAM

yolo = YOLO("yolo11l.pt")                              # COCO 사전학습: 캔 클래스가 없다
yoloworld = YOLOWorld("yolov8s-worldv2.pt")            # 문장으로 찾을 대상을 지정
yoloworld.set_classes(["bottle", "can"])
sam = SAM("sam2.1_b.pt")                               # 프롬프트 없이 모든 것을 분할

DEV_ARG = 0 if DEVICE == "cuda" else "cpu"

def detect_yolo(fn):
    r = yolo.predict(f"data/photos/{fn}.jpg", conf=0.05, iou=0.7, imgsz=1280, device=DEV_ARG, verbose=False)[0]
    b = r.boxes
    return b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy().astype(int)

def detect_world(fn):
    r = yoloworld.predict(f"data/photos/{fn}.jpg", conf=0.05, iou=0.7, imgsz=1280, device=DEV_ARG, verbose=False)[0]
    b = r.boxes
    return b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy().astype(int)

def detect_sam(fn, max_side=1536):
    im = load_img(fn)
    W, H = im.size
    s = max_side / max(W, H)
    small = np.array(im.resize((int(W * s), int(H * s))))
    r = sam.predict(small, device=DEV_ARG, verbose=False)[0]
    if r.masks is None:
        return np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)
    boxes = []
    for mk in r.masks.data:                            # 마스크를 하나씩 상자로 바꿔 메모리를 아낀다
        mk = mk.bool()
        rows, cols = torch.where(mk.any(1))[0], torch.where(mk.any(0))[0]
        if len(rows) == 0 or len(cols) == 0:
            boxes.append([0, 0, 0, 0])
        else:
            boxes.append([cols[0].item(), rows[0].item(), cols[-1].item() + 1, rows[-1].item() + 1])
    boxes = np.array(boxes, float)
    mh, mw = r.masks.data.shape[1:]
    boxes[:, [0, 2]] *= W / mw
    boxes[:, [1, 3]] *= H / mh
    return boxes, np.ones(len(boxes)), np.zeros(len(boxes), int)

CANDIDATES = {
    "A":  dict(name="YOLO11 + SigLIP 2",       detect=detect_yolo,  keep_options=[[39], [39, 41], [39, 41, 75]]),  # COCO: bottle, cup, vase
    "A2": dict(name="YOLO-World + SigLIP 2",   detect=detect_world, keep_options=[[0, 1], [0], [1]]),             # 0: bottle, 1: can
    "B":  dict(name="SAM 2 + SigLIP 2",        detect=detect_sam,   keep_options=[[0]]),
}

## 4. 용기 후보 → 최종 개수
1. `prepare`: 사진마다 상자를 찾고 잘라낸 그림을 SigLIP 2로 채점해 **캐시**한다(한 번만 계산).
2. `decide`: 캐시에 임계값을 적용해 최종 개수를 만든다. 임계값만 바꿀 때는 모델을 다시 돌리지 않는다.

In [ ]:
LOOSE = dict(min_area=0.0003, max_area=0.3, min_aspect=0.7, max_aspect=10.0)   # 채점 전 느슨한 거름
CACHE = {}

def geometry_ok(boxes, W, H, P):
    w, h = boxes[:, 2] - boxes[:, 0], boxes[:, 3] - boxes[:, 1]
    area = w * h / (W * H)
    asp = h / np.maximum(w, 1e-6)
    return (w > 0) & (h > 0) & (area >= P["min_area"]) & (area <= P["max_area"]) & \
           (asp >= P["min_aspect"]) & (asp <= P["max_aspect"])

def prepare(cand, fn):
    key = (cand, fn)
    if key in CACHE:
        return CACHE[key]
    im = load_img(fn)
    W, H = im.size
    t0 = time.time()
    boxes, det, cls = CANDIDATES[cand]["detect"](fn)
    t_det = time.time() - t0
    ok = geometry_ok(boxes, W, H, LOOSE) & (det >= 0.05)
    boxes, det, cls = boxes[ok], det[ok], cls[ok]
    t0 = time.time()
    crops = []
    for x0, y0, x1, y1 in boxes:
        pw, ph = (x1 - x0) * 0.05, (y1 - y0) * 0.05
        crops.append(im.crop((max(0, x0 - pw), max(0, y0 - ph), min(W, x1 + pw), min(H, y1 + ph))))
    scores = classify(crops)
    t_cls = time.time() - t0
    CACHE[key] = dict(boxes=boxes, det=det, cls=cls, scores=scores, W=W, H=H, t_det=t_det, t_cls=t_cls)
    return CACHE[key]

DEFAULT_P = dict(conf=0.1, tau=0.1, nms_iou=0.5, min_area=0.0008, max_area=0.2,
                 min_aspect=1.1, max_aspect=6.0, keep=[0])

def decide(item, P):
    """캐시된 후보에 임계값을 적용해 [(상자, 라벨, 점수)]를 돌려준다."""
    if len(item["boxes"]) == 0:
        return []
    m = (item["det"] >= P["conf"]) & np.isin(item["cls"], P["keep"]) & \
        geometry_ok(item["boxes"], item["W"], item["H"], P)
    idx = np.where(m)[0]
    if len(idx) == 0:
        return []
    rank = item["det"][idx] * item["scores"][idx].max(1)
    kept = torchvision.ops.nms(torch.tensor(item["boxes"][idx], dtype=torch.float32),
                               torch.tensor(rank, dtype=torch.float32), P["nms_iou"]).numpy()
    out = []
    for i in idx[kept]:
        s = item["scores"][i]
        k = int(s.argmax())
        lab = CLASSES[k]
        if lab == "bg":
            continue
        if lab in PRODUCTS and s[k] < P["tau"]:
            lab = "other"
        out.append((item["boxes"][i], lab, float(s[k])))
    return out

def predict_counts(cand, files, P):
    pred = defaultdict(dict)
    for fn in files:
        c = defaultdict(int)
        for _, lab, _ in decide(prepare(cand, fn), P):
            c[lab] += 1
        pred[fn] = dict(c)
    return pred

## 5. 개발셋에서 값 정하기 (평가셋은 보지 않는다)
`conf`, `tau`, `nms_iou`, `min_aspect`, `keep`을 격자로 바꿔 보며 **개발셋 8장**에서 가장 나은 값을 고른다. 기준: S2(제품별 ±1 비율) → S1(총 개수 맞은 사진 수) → S5(8종 과다 집계, 적을수록 좋음) 순서.

**튜닝 기록**: 1차(conf 0.05~0.3, tau 0.02~0.2)에서 고른 값이 격자의 끝(conf 0.3, tau 0.2, B는 tau 0.02)에 걸려 **2차로 격자를 넓혔다.** 둘 다 개발셋만 썼고 평가셋은 보지 않았다. 1차 결과는 아래 코드의 `PREVIOUS_ROUND`와 `모델선정계획.md`에 남겨 두었다.

In [ ]:
# 2차 튜닝: 1차(좁은 격자)에서 최적값이 격자 끝에 걸려 격자를 넓혔다. 평가셋은 아직 보지 않았다.
GRID = dict(conf=[0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6],
            tau=[0.005, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5],
            nms_iou=[0.4, 0.5, 0.6], min_aspect=[0.9, 1.3])
PREVIOUS_ROUND = {   # 1차 결과(개발셋 8장, conf 0.05~0.3, tau 0.02~0.2)
    "A":  dict(conf=0.3,  tau=0.2,  dev_S2=0.556, dev_S1_ok=3, dev_S5_proxy=27),
    "A2": dict(conf=0.2,  tau=0.2,  dev_S2=0.722, dev_S1_ok=2, dev_S5_proxy=21),
    "B":  dict(conf=0.05, tau=0.02, dev_S2=0.700, dev_S1_ok=2, dev_S5_proxy=38),
}

def dev_objective(pred):
    r = ev.score(SEL["dev"], GT, pred, include_other=True)
    return (r["S2"]["rate"] or 0.0, r["S1"]["ok"], -r["S5_over_count_proxy"]), r

CHOSEN = {}
for cand, spec in CANDIDATES.items():
    print(f"\n=== {cand}: {spec['name']} — 개발셋 {len(SEL['dev'])}장 준비 ===")
    for fn in SEL["dev"]:
        prepare(cand, fn)
    best, best_P, tried = None, None, 0
    for keep in spec["keep_options"]:
        for conf, tau, nms_iou, min_aspect in itertools.product(*GRID.values()):
            P = dict(DEFAULT_P, keep=keep, conf=conf, tau=tau, nms_iou=nms_iou, min_aspect=min_aspect)
            obj, _ = dev_objective(predict_counts(cand, SEL["dev"], P))
            tried += 1
            if best is None or obj > best:
                best, best_P = obj, P
    CHOSEN[cand] = dict(params={k: (v if not isinstance(v, np.generic) else v.item()) for k, v in best_P.items()},
                        dev_S2=best[0], dev_S1_ok=best[1], dev_S5_proxy=-best[2], combos_tried=tried)
    print(json.dumps(CHOSEN[cand], ensure_ascii=False))

os.makedirs("results", exist_ok=True)
with open("results/chosen_params.json", "w", encoding="utf-8") as f:
    json.dump(dict(run_id=RUN_ID, tuning_round=2, previous_round=PREVIOUS_ROUND, env=ENV,
                   siglip=SIGLIP_ID, grid=GRID, prompts=PROMPTS, chosen=CHOSEN),
              f, ensure_ascii=False, indent=1)
print("\n저장: results/chosen_params.json  ← 이 값은 이제 바꾸지 않는다")

## 6. 고정한 값으로 20장 실행 → 평가셋 판정
여기서부터 값은 고정이다. 평가셋 점수가 마음에 들지 않아도 값을 다시 정하지 않는다(다시 하려면 새 실행으로 기록하고 이유를 남긴다).

In [ ]:
PALETTE = {"P1": "#16a34a", "P2": "#84cc16", "P3": "#2563eb", "P4": "#0d9488", "P5": "#dc2626",
           "P6": "#111827", "P7": "#f59e0b", "P8": "#db2777", "other": "#6b7280"}

def run_final(cand):
    P = CHOSEN[cand]["params"]
    out_dir = f"results/{cand}/{RUN_ID}"
    os.makedirs(out_dir + "/vis", exist_ok=True)
    rows, timing = [], []
    for fn in SEL["all"]:
        item = prepare(cand, fn)
        dets = decide(item, P)
        c = defaultdict(int)
        for _, lab, _ in dets:
            c[lab] += 1
        rows += [(fn, p, n) for p, n in sorted(c.items())]
        timing.append((fn, item["t_det"], item["t_cls"]))
        im = load_img(fn)
        d = ImageDraw.Draw(im)
        lw = max(3, im.width // 600)
        for (x0, y0, x1, y1), lab, sc in dets:
            col = PALETTE.get(lab, "#6b7280")
            d.rectangle([x0, y0, x1, y1], outline=col, width=lw)
            d.text((x0 + 4, y0 + 2), f"{lab} {sc:.2f}", fill=col)
        im.thumbnail((1600, 1600))
        im.save(f"{out_dir}/vis/{fn}.jpg", quality=85)
    with open(f"{out_dir}/pred.csv", "w", encoding="utf-8", newline="") as f:
        f.write("file,product,count\n" + "".join(f"{a},{b},{n}\n" for a, b, n in rows))
    with open(f"{out_dir}/timing.csv", "w", encoding="utf-8", newline="") as f:
        f.write("file,detect_seconds,classify_seconds\n" + "".join(f"{a},{b:.4f},{c:.4f}\n" for a, b, c in timing))
    with open(f"{out_dir}/params.json", "w", encoding="utf-8") as f:
        json.dump(dict(candidate=cand, name=CANDIDATES[cand]["name"], params=P, env=ENV, siglip=SIGLIP_ID), f,
                  ensure_ascii=False, indent=1)
    return out_dir

OUT = {}
for cand in CANDIDATES:
    print(f"\n=== {cand}: {CANDIDATES[cand]['name']} — 20장 ===")
    OUT[cand] = run_final(cand)
    r = subprocess.run([sys.executable, "scripts/evaluate.py", f"{OUT[cand]}/pred.csv", "--out", f"{OUT[cand]}/metrics.json"],
                       capture_output=True, text=True)
    print(r.stdout[-2500:] if r.returncode == 0 else r.stderr)

## 7. 후보 비교표

In [ ]:
import pandas as pd

rows = []
for cand, out in OUT.items():
    m = json.load(open(f"{out}/metrics.json", encoding="utf-8"))
    t = pd.read_csv(f"{out}/timing.csv")
    t = t[t.file.isin(SEL["eval"])]
    rows.append({
        "후보": f"{cand} {CANDIDATES[cand]['name']}",
        "평가셋 S1(총 개수, /12)": m["eval"]["S1"]["ok"],
        "평가셋 S2(제품별)": m["eval"]["S2"]["rate"],
        "평가셋 S4(발주)": m["eval"]["S4"]["rate"],
        "평가셋 S5(과다, 대략치)": m["eval"]["S5_over_count_proxy"],
        "개발셋 S2": m["dev"]["S2"]["rate"],
        "모델 시간 12장 합계(초)": round(float((t.detect_seconds + t.classify_seconds).sum()), 1),
        "S1 통과": m["pass"]["S1"], "S2 통과": m["pass"]["S2"], "S4 통과": m["pass"]["S4"],
    })
table = pd.DataFrame(rows)
table.to_csv(f"results/compare_{RUN_ID}.csv", index=False)
table

> 위 "모델 시간"은 사진을 받은 뒤 모델이 일하는 시간이다. 문제정의서의 S3(시간)는 여기에 **사람이 결과 이미지를 보고 틀린 개수를 고치는 시간**을 더해서 따로 잰다. 첫 사진은 모델 예열 시간이 섞여 있어 느리게 나올 수 있다.

## 8. 틀린 사례 보기

In [ ]:
from IPython.display import display

def show(cand, fn):
    display(Image.open(f"{OUT[cand]}/vis/{fn}.jpg"))

# 예: 평가셋에서 가장 틀린 칸이 많은 사진 하나를 골라 후보별로 비교
for cand, out in OUT.items():
    m = json.load(open(f"{out}/metrics.json", encoding="utf-8"))
    print(cand, [(c["file"], c["product"], c["gt"], c["pred"]) for c in m["eval"]["wrong_cells"]][:8])
# show("B", "IMG_4206")

## 9. 결과 내려받기

In [ ]:
shutil.make_archive("results_bundle", "zip", "results")
try:
    from google.colab import files
    files.download("results_bundle.zip")
except Exception as e:
    print("Colab이 아니면 results_bundle.zip을 직접 가져가세요:", e)

## 10. 선정 근거 (결과를 본 뒤 직접 적는다)

| 항목 | 내용 |
|---|---|
| 평가셋 결과 | (비교표를 붙여넣기) |
| 가장 나은 후보와 이유 | |
| 다른 후보가 못한 이유(실패 사례 사진 번호) | |
| 라이선스 확인(모델 카드, 확인 날짜) | |
| 한계와 다음에 바꿔 볼 것 | |

기준을 못 넘었다면 그 이유를 적는 것도 결과다.